# Live Camera Display and HUD with Pygame

## 1. Overview

This project demonstrates how to capture live images from a webcam and display them using Pygame.

It also introduces how to draw text, rectangles and transparent overlays on top of a live camera feed.

The implementation combines three libraries:

- **OpenCV:** Captures images from the webcam.
- **NumPy:** Handles image data as multidimensional arrays.
- **Pygame:** Displays images and draws graphical elements.

### Project objectives

- Capture live camera frames.
- Convert images into Pygame surfaces.
- Display the camera feed in a Pygame window.
- Render text on top of images.
- Create transparent surfaces.
- Draw graphical elements such as rectangles and lines.
- Combine multiple surfaces into one display.

---

## 2. Required libraries

```python
import pygame
import cv2
import numpy as np
```

| Library | Responsibility |
|---|---|
| `pygame` | Window management, graphics, text and input handling |
| `cv2` | Webcam access and image conversion |
| `numpy` | Image array manipulation |

Install the required packages:

```bash
pip install pygame opencv-python numpy
```

---

## 3. Initialize the application

### Pygame initialization

```python
pygame.init()
```

Initializes the imported Pygame modules.

Before creating windows, fonts or other Pygame resources, we should initialize Pygame.

### Initialize the webcam

```python
camera = cv2.VideoCapture(0)
```

`cv2.VideoCapture()` initializes a video capture device.

**Parameter:**

- `0`: Default webcam.
- `1`: Second available camera.
- Other indices can represent additional cameras.

The returned object provides methods to capture video frames.

### Create the Pygame window

```python
WIDTH = 800
HEIGHT = 600

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Live Camera HUD")
```

`pygame.display.set_mode()` creates the main application window.

Parameters:

- `WIDTH`: Window width in pixels.
- `HEIGHT`: Window height in pixels.

The returned object, `screen`, is a Pygame Surface representing the display window.

`pygame.display.set_caption()` sets the window title.

### Create a clock

```python
clock = pygame.time.Clock()
```

Creates a Clock object used to control the application's frame rate.

For example:

```python
clock.tick(30)
```

Limits the application to approximately 30 frames per second.

---

## 4. Pygame fonts

```python
title_font = pygame.font.SysFont("Arial", 25, bold=True)
font = pygame.font.SysFont("Arial", 18)
```

`pygame.font.SysFont()` creates a font object using a system-installed font.

Parameters:

| Parameter | Description |
|---|---|
| `Arial` | Font family |
| `25` | Font size in pixels |
| `bold=True` | Enables bold formatting |

### Rendering text

```python
text = font.render(
    "LIVE CAMERA",
    True,
    (255, 255, 255)
)
```

`font.render()` converts a string into a Pygame Surface containing the rendered text.

Parameters:

1. Text to display.
2. Anti-aliasing (`True` enables smoother text).
3. Text color in RGB format.

The returned Surface can be displayed using `blit()`.

---

## 5. Understanding Pygame Surfaces

A Surface represents an image or drawable area in Pygame.

It can contain:
- Camera images.
- Text.
- Rectangles.
- Lines.
- Other graphical elements.

In this project, we use two important surfaces:

```python
screen
```

The main application window.

```python
hud
```

A separate surface used to draw the HUD.

### Creating a transparent surface

```python
hud = pygame.Surface(
    (350, 130),
    pygame.SRCALPHA
)
```

`pygame.Surface()` creates a new drawable surface.

Parameters:

- `(350, 130)`: Width and height.
- `pygame.SRCALPHA`: Enables per-pixel alpha transparency.

### Understanding RGBA

A normal RGB color contains three values:

```python
(255, 0, 0)
```

An RGBA color contains four:

```python
(255, 0, 0, 170)
```

The fourth value represents alpha (transparency).

| Alpha | Appearance |
|---|---|
| 0 | Completely transparent |
| 100 | Highly transparent |
| 170 | Semi-transparent |
| 255 | Completely opaque |

To fill the HUD surface:

```python
hud.fill((20, 20, 20, 170))
```

This creates a dark, semi-transparent background.

---

## 6. Capturing live camera images

Inside the main loop:

```python
ret, frame = camera.read()
```

`camera.read()` captures a new frame from the webcam.

It returns two values:

| Variable | Description |
|---|---|
| `ret` | Boolean indicating whether frame capture succeeded |
| `frame` | Captured image represented as a NumPy array |

Check whether the frame was captured successfully:

```python
if ret:
    # Process image
```

### Understanding image dimensions

A captured image is represented as a NumPy array with the following structure:

```python
(height, width, channels)
```

For a 640 × 480 RGB image:

```python
(480, 640, 3)
```

The three channels represent red, green and blue.

---

## 7. Converting BGR to RGB

OpenCV captures images using BGR channel ordering.

Pygame expects RGB ordering when creating a surface from a standard three-channel array.

Convert the image:

```python
frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
```

`cv2.cvtColor()` converts an image from one color space to another.

Parameters:

- `frame`: Source image.
- `cv2.COLOR_BGR2RGB`: Conversion from BGR to RGB.

---

## 8. Converting NumPy arrays into Pygame surfaces

```python
surface = pygame.surfarray.make_surface(
    np.transpose(frame, (1, 0, 2))
)
```

This operation involves two functions.

### A. np.transpose()

```python
np.transpose(frame, (1, 0, 2))
```

Changes the order of the array axes.

The camera image initially has this structure:

```python
(height, width, channels)
```

However, `pygame.surfarray.make_surface()` expects:

```python
(width, height, channels)
```

Therefore, we transpose the first two axes.

The `(1, 0, 2)` argument specifies the new axis ordering:

- Axis 1 becomes the first axis.
- Axis 0 becomes the second axis.
- Axis 2 remains unchanged.

### B. pygame.surfarray.make_surface()

```python
pygame.surfarray.make_surface(array)
```

Converts a NumPy array into a Pygame Surface.

This allows us to use Pygame functions such as `blit()` to display the image.

### Resize the image

```python
surface = pygame.transform.smoothscale(
    surface,
    (WIDTH, HEIGHT)
)
```

`pygame.transform.smoothscale()` resizes a Surface using filtered scaling.

Parameters:

- `surface`: Original Surface.
- `(WIDTH, HEIGHT)`: Desired dimensions.

This allows the camera image to fit the Pygame window.

---

## 9. Displaying images using blit()

```python
screen.blit(surface, (0, 0))
```

`blit()` copies one Surface onto another.

Syntax:

```python
destination.blit(source, position)
```

In this example:

- Destination: `screen`.
- Source: `surface`.
- Position: `(0, 0)`.

The image is displayed at the top-left corner of the Pygame window.

### Important: Rendering order

Pygame draws surfaces in the order they are blitted.

For example:

```python
screen.blit(surface, (0, 0))
screen.blit(hud, (20, 20))
```

First, the camera image is displayed.

Then, the HUD is displayed on top of the camera.

This is important when creating graphical overlays.

---

## 10. Drawing text on a HUD surface

First, render the text:

```python
title = title_font.render(
    "LIVE CAMERA",
    True,
    (255, 255, 255)
)
```

Then draw it onto the HUD:

```python
hud.blit(title, (15, 15))
```

Notice that we are using `blit()` again.

This time:

- Destination: `hud`.
- Source: `title`.
- Position: `(15, 15)`.

We can create multiple text elements:

```python
subtitle = font.render(
    "Computer Vision Demo",
    True,
    (200, 200, 200)
)

hud.blit(subtitle, (15, 65))
```

---

## 11. Drawing graphical elements

Pygame provides functions for drawing shapes.

### Drawing a line

```python
pygame.draw.line(
    hud,
    (255, 140, 0),
    (15, 105),
    (335, 105),
    2
)
```

Syntax:

```python
pygame.draw.line(
    surface,
    color,
    start_position,
    end_position,
    width
)
```

This creates a horizontal orange separator.

### Drawing a rectangle

```python
pygame.draw.rect(
    hud,
    (255, 140, 0),
    (15, 15, 200, 50),
    2
)
```

Syntax:

```python
pygame.draw.rect(
    surface,
    color,
    (x, y, width, height),
    thickness
)
```

A thickness of `0` or a negative value creates a filled rectangle. A positive value creates an outline.

---

## 12. Understanding the main application loop

The application needs to continuously capture frames, process images and update the display.

```python
running = True

while running:

    for event in pygame.event.get():

        if event.type == pygame.QUIT:
            running = False

    ret, frame = camera.read()

    if ret:
        # Process camera image
        # Draw camera image
        # Draw HUD

    pygame.display.flip()

    clock.tick(30)
```

### pygame.event.get()

Retrieves events from the Pygame event queue.

Events include:

- Window closing.
- Keyboard input.
- Mouse input.

### pygame.QUIT

Identifies the event generated when the user closes the application window.

### pygame.display.flip()

Updates the entire display window.

It makes the latest drawing operations visible.

### clock.tick(30)

Limits the loop to approximately 30 iterations per second.

---


## 13. Important concepts to remember

| Function | Purpose |
|---|---|
| `pygame.init()` | Initializes Pygame |
| `cv2.VideoCapture()` | Initializes a camera |
| `camera.read()` | Captures a camera frame |
| `cv2.cvtColor()` | Converts image color channels |
| `np.transpose()` | Changes NumPy array axis ordering |
| `pygame.surfarray.make_surface()` | Converts an array into a Pygame Surface |
| `pygame.transform.smoothscale()` | Resizes a Surface |
| `pygame.Surface()` | Creates a drawable surface |
| `pygame.SRCALPHA` | Enables per-pixel transparency |
| `pygame.font.SysFont()` | Creates a font object |
| `font.render()` | Converts text into a Surface |
| `pygame.draw.line()` | Draws a line |
| `pygame.draw.rect()` | Draws a rectangle |
| `blit()` | Draws one Surface onto another |
| `pygame.display.flip()` | Updates the display |
| `pygame.event.get()` | Retrieves application events |
| `pygame.time.Clock()` | Creates a frame-rate controller |
| `clock.tick()` | Limits the frame rate |
| `camera.release()` | Releases the camera resource |
| `pygame.quit()` | Shuts down Pygame |

## 14. Suggested exercises

1. Change the HUD position by modifying the coordinates in `screen.blit(hud, (20, 20))`.
2. Experiment with different alpha values in `hud.fill()`.
3. Display the camera FPS using `clock.get_fps()`.
4. Create a second transparent HUD surface.
5. Draw a rectangle around the camera image.
6. Display the current mouse coordinates using Pygame events.

**Key takeaway:** A Pygame Surface is the central concept behind this implementation. The camera image, rendered text and HUD are all represented as surfaces. By combining them with `blit()`, you can build complex visual interfaces using relatively simple drawing operations.

In [1]:
import cv2
import pygame
import numpy as np
import time

pygame 2.6.0 (SDL 2.28.4, Python 3.12.10)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [ ]:
pygame.init()

# Start webcam

camera = cv2.VideoCapture(0)

# Window Dimentions

WIDTH = 800
HEIGHT = 600

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Pygame - Live Camera")

clock = pygame.time.Clock()

running = True

while running:

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    ret, frame = camera.read()

    if ret:

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        surface = pygame.surfarray.make_surface(
            np.transpose(frame, (1,0,2))
        )

        surface = pygame.transform.smoothscale(
            surface,
            (WIDTH, HEIGHT)
        )

        screen.blit(surface, (0,0))

    pygame.display.flip()
    clock.tick(30)

camera.release()
pygame.quit()

In [ ]:
import pygame
import cv2
import numpy as np

In [ ]:
pygame.init()
camera = cv2.VideoCapture(0)

WIDTH = 800
HEIGHT = 600

In [ ]:
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Pygame - Learning")
clock = pygame.time.Clock()

title_font = pygame.font.SysFont("Arial", 25, bold=True)
font = pygame.font.SysFont("Arial", 18)

hud = pygame.surface(
    (350, 130),
    pygame.SRCALPHA
)

running = True

while running:

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
    #Ret =  return
    ret, frame = camera.read()

    if ret:

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        surface = pygame.surfarray.make_surface(
            np.transpose(frame, (1,0,2))
        )

        surface = pygame.transform.smoothscale(
            surface,
            (WIDTH, HEIGHT)
        )

        screen.blit(surface, (0,0))
        hud.fill((20, 20, 20, 170))
         # Render text
#         title = title_font.render(
#             "LIVE CAMERA",
#             True,
#             (255, 255, 255)
#         )
#         subtitle = font.render(
#             f"Resolution: {frame.shape[1]} x {frame.shape[0]}",
#             True,
#             (200, 200, 200)
#         )


#         # Draw text on HUD
#         hud.blit(title, (15, 15))
#         hud.blit(subtitle, (15, 65))

#         # Draw separator
#         pygame.draw.line(
#             hud,
#             (255, 140, 0),
#             (15, 105),
#             (335, 105),
#             2
#         )

#         # Draw HUD on main screen
#         screen.blit(hud, (20, 20))

#     pygame.display.flip()

#     clock.tick(30)

# # Release resources
# camera.release()
# pygame.quit()




